In [ ]:
import json
from pathlib import Path

import pandas as pd
import seaborn as sns
from common import get_color, get_method_label
from matplotlib import pyplot as plt

In [ ]:
log_root = Path("../")
files = [
    "log/zeus/cifar100/ball/00/zeus_monitor.json",
    "log/zeus/cifar100/clora/00/zeus_monitor.json",
    "log/zeus/cifar100/ewc/00/zeus_monitor.json",
    "log/zeus/cifar100/inflora/00/zeus_monitor.json",
    "log/zeus/cifar100/lora/00/zeus_monitor.json",
    "log/zeus/cifar100/rwalk/00/zeus_monitor.json",
    "log/zeus/cifar100/sdlora/00/zeus_monitor.json",
    "log/zeus/cifar100/tball/00/zeus_monitor.json",
    "log/zeus/cifar100/tball-mnd/00/zeus_monitor.json",
]
dfs = []
for file in files:
    method = file.split("/")[3]
    with open(log_root / file) as f:
        data = json.load(f)

    df = pd.DataFrame(pd.json_normalize(data["summary"]))
    df["method"] = method
    dfs.append(df)

df = pd.concat(dfs, ignore_index=True)

In [ ]:
train_exp_size = 5000
eval_exp_size = 1000

df[["method", "train_experience.total_gpu_energy", "eval_experience.total_gpu_energy"]]
df["eval_experience.energy_per_example"] = df["eval_experience.total_energy"] / (
    df["eval_experience.count"] * eval_exp_size
)
df["train_experience.energy_per_example"] = df["train_experience.total_energy"] / (
    df["train_experience.count"] * train_exp_size
)
df["train.time_per_example"] = df["train_experience.time"] / (
    df["train_experience.count"] * train_exp_size
)
df["eval.time_per_example"] = df["eval_experience.time"] / (
    df["eval_experience.count"] * eval_exp_size
)

In [ ]:
df_melted = df.melt(
    id_vars="method",
    value_vars=[
        "train_experience.energy_per_example",
        "eval_experience.energy_per_example",
    ],
    var_name="experience",
    value_name="energy_per_example",
)
ax = sns.barplot(data=df_melted, x="method", y="energy_per_example", hue="experience")

# Add lebales
ax.set_xlabel("Method")
ax.set_ylabel("Energy per Example (Joules)")

In [ ]:
import scienceplots
from matplotlib.axes import Axes


def joules_to_kwh(joules):
    return joules / (3600 * 1000)


if scienceplots is not None:
    plt.style.use(["science", "nature"])

width = 397.48 / 72.27
aspect = 3
height = width / aspect
figsize = (width, height)

fig, axes = plt.subplots(1, 3, figsize=figsize, constrained_layout=True)

ax0: Axes = axes[0]
ax1: Axes = axes[1]
ax3: Axes = axes[2]

labels = df["method"].apply(get_method_label)
colors = df["method"].apply(get_color)

energy_per_example = df["train_experience.energy_per_example"]
energy_per_example_eval = df["eval_experience.energy_per_example"]

n_tasks = 10
n_epochs = 20
total_energy = joules_to_kwh(
    energy_per_example * train_exp_size * n_tasks * n_epochs
    + energy_per_example_eval * eval_exp_size * n_tasks**2
)

sort_train = energy_per_example.argsort()
sort_eval = energy_per_example_eval.argsort()
sort_total = total_energy.argsort()

ax0.barh(labels[sort_train], energy_per_example[sort_train], color=colors[sort_train])
ax1.barh(labels[sort_eval], energy_per_example_eval[sort_eval], color=colors[sort_eval])
ax3.barh(labels[sort_total], total_energy[sort_total], color=colors[sort_total])

ax0.set_xlabel("Energy per Example (Joules)")
ax0.set_title("Training Energy per Example")
ax1.set_xlabel("Energy per Example (Joules)")
ax1.set_title("Evaluation Energy per Example")

ax3.set_xlabel("Total Energy (kWh)")
ax3.set_title("Total Energy")

# Remove y ticks
for ax in axes:
    ax.tick_params(axis="y", which="both", left=False, right=False)

fig.savefig("plots/energy-usage.pdf")

In [ ]:
(train_exp_size * n_tasks * n_epochs) / (eval_exp_size * n_tasks * n_tasks)

In [ ]:
eval_exp_size * n_tasks * n_tasks